# 2  Predicting processing time

**Goal:** predict how many days a visa application will take, check that the model beats a simple guess, and find out which inputs matter.

**Data:** the prepared visa applications from exercise 1.

**The data used here:**
- `visa_applications_clean.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/visa_applications_clean.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/visa_applications_clean.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

## Step 1  Inputs and target

In [ ]:
#@title Step 1  Load the prepared applications
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error as MAE, mean_squared_error
df = pd.read_csv(DATA + "visa_applications_clean.csv")
INPUTS = ["visa_type", "applicant_region", "channel", "sponsor_type", "applicant_age", "previous_visits", "documents_complete", "additional_document_review_required", "supporting_document_count"]
X = pd.get_dummies(df[INPUTS], dtype=int); y = df["processing_days"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
table(df[INPUTS + ["processing_days"]], {(i, "processing_days"): "#fbd9d9" for i in range(8)}, n=8)
box("What you see", "Nine <b>inputs</b> that are known when an application arrives, and the <b>target</b> in red: the processing time in days. "
    f"The model learns from {len(X_train):,} past applications and is tested on {len(X_test):,} it has never seen.", "info")

## Step 2  The simplest guess

In [ ]:
#@title Step 2  Always predict the average
avg = y_train.mean(); base_mae = MAE(y_test, np.full(len(y_test), avg))
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.scatter(range(150), y_test.values[:150], s=18, color=INK); ax.axhline(avg, color=RED, lw=3)
for i, v in enumerate(y_test.values[:150]): ax.plot([i, i], [v, avg], color=RED, alpha=.3)
ax.set_xlabel("150 test applications"); ax.set_ylabel("days"); ax.set_title("The average as a prediction: every red line is a miss"); plt.show()
big((f"{avg:.1f} days", "average processing time"), (f"{base_mae:.1f} days", "average miss (MAE)", RED))
box("Remember this number", f"Guessing the average misses by {base_mae:.1f} days. A model is only useful if it does clearly better.", "see")

## Step 3  Two models

In [ ]:
#@title Step 3  Train a linear regression and a regression tree
lin = LinearRegression().fit(X_train, y_train); tree = DecisionTreeRegressor(max_depth=5, random_state=42).fit(X_train, y_train)
res = {"Baseline: the average": base_mae, "Linear regression": MAE(y_test, lin.predict(X_test)), "Regression tree": MAE(y_test, tree.predict(X_test))}
bars(res, "Average miss on the test data (days)", highlight="Baseline: the average", fmt="{:.2f} days")
best = min(res, key=res.get)
box("Do you see it?", f"Both models miss far less than the average. The {best.lower()} misses by {res[best]:.1f} days instead of {base_mae:.1f}. "
    "<b>Now the model can tell an applicant roughly how long the application will take</b>, and the service can see which applications will be slow.", "see")

In [ ]:
#@title Step 3b  Which metric punishes large delays more?  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "MAE: the average miss", "RMSE: the root of the average squared miss"]
p = lin.predict(X_test)
big((f"{MAE(y_test, p):.2f}", "MAE (days)"), (f"{mean_squared_error(y_test, p) ** 0.5:.2f}", "RMSE (days)"))
_ = answer(question, {
    "MAE: the average miss": (False, "MAE counts a 10-day miss ten times as much as a 1-day miss: exactly in proportion."),
    "RMSE: the root of the average squared miss": (True, "RMSE squares each miss first, so a 10-day miss counts a hundred times as much as a 1-day miss. Use it when long delays hurt most."),
})

## Step 4  How deep should the tree grow?

In [ ]:
#@title Step 4  Move the slider  (then move the slider)
depths = range(1, 21)
tr = [MAE(y_train, DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train).predict(X_train)) for d in depths]
te = [MAE(y_test, DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train).predict(X_test)) for d in depths]
def show(depth):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(depths, tr, color=INK, lw=3, label="error on training data"); ax.plot(depths, te, color=RED, lw=3, label="error on test data")
    ax.scatter([depth, depth], [tr[depth - 1], te[depth - 1]], s=120, color=[INK, RED], zorder=3)
    ax.set_xlabel("depth of the tree"); ax.set_ylabel("MAE (days)"); ax.legend(); plt.show()
    big((depth, "depth"), (f"{tr[depth - 1]:.2f}", "training error"), (f"{te[depth - 1]:.2f}", "test error", RED))
    if depth >= 12: box("Do you see it?", "The training error keeps falling, but the test error rises again. The deep tree memorises the training data instead of learning the pattern.", "see")
_ = live(show, depth=islider("Depth of the tree", 1, 20, 3))

In [ ]:
#@title Step 4b  Your choice  (choose, then click &#9654;)
choice = "Choose..." #@param ["Choose...", "Depth 2", "Depth 8", "Depth 20"]
_ = answer(choice, {
    "Depth 2": (False, "Too simple: it misses patterns, so both errors are still high."),
    "Depth 8": (True, "Around here the test error is lowest; depths from about 5 to 9 are almost equally good. Deeper trees only get better on data they have already seen."),
    "Depth 20": (False, "Near perfect on the training data and worse on new applications: the tree has memorised the training data."),
})

## Step 5  Which inputs matter?

In [ ]:
#@title Step 5  Switch inputs off and see what happens  (untick inputs on the right, then click &#9654;)
visa_type = True #@param {type:"boolean"}
applicant_region = True #@param {type:"boolean"}
channel = True #@param {type:"boolean"}
sponsor_type = True #@param {type:"boolean"}
applicant_age = True #@param {type:"boolean"}
previous_visits = True #@param {type:"boolean"}
documents_complete = True #@param {type:"boolean"}
additional_document_review_required = True #@param {type:"boolean"}
supporting_document_count = True #@param {type:"boolean"}
keep = [c for c in INPUTS if globals()[c]]
def mae_with(cols):
    Xk = pd.get_dummies(df[cols], dtype=int); a, b, ya, yb = train_test_split(Xk, y, test_size=0.2, random_state=42)
    return MAE(yb, DecisionTreeRegressor(max_depth=8, random_state=42).fit(a, ya).predict(b))
full, mine = mae_with(INPUTS), mae_with(keep) if keep else base_mae
big((f"{full:.2f}", "MAE with all nine inputs"), (f"{mine:.2f}", f"MAE with your {len(keep)} inputs", GREEN if mine <= full + 0.05 else RED))
drop = pd.Series({c: mae_with([k for k in INPUTS if k != c]) for c in INPUTS}).sort_values(ascending=False)
bars(drop, "MAE after removing one input at a time (days)", fmt="{:.2f}")
box("Do you see it?", f"Removing <b>{drop.index[0]}</b> makes the error jump to {drop.iloc[0]:.2f} days, so it matters. "
    f"Removing <b>{drop.index[-1]}</b> changes almost nothing, so it can go. Some inputs overlap: when one is removed, another that carries similar information takes its place, so the error hardly changes. "
    "Untick inputs above and click &#9654; to test your own selection.", "see")

## Result

In [ ]:
#@title Result
box("Do you see it now?", f"From nine facts known on arrival, the {best.lower()} predicts the processing time to within about {res[best]:.1f} days on average, against {base_mae:.1f} days for a plain guess. "
    "<b>The service can now tell applicants what to expect and spot slow applications early.</b> You also saw the two checks that keep it honest: "
    "test on data the model has not seen, and do not let the tree grow until it memorises.", "see")

---
## Stretch (optional, for those who code)

In [ ]:
# Stretch: try a RandomForestRegressor(n_estimators=200) and compare its MAE
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)
MAE(y_test, forest.predict(X_test))